# Day 32 — Data pipeline: augment train, never val/test

Status: COMPLETE. Train gets Resize(224) + flip/rotation/brightness + ImageNet
normalize; val/test get deterministic Resize + normalize. Stratified val carved
from train (official 16-image val unusable). 6/6 tests pass.

In [1]:
import sys
sys.path.insert(0, "..")
import torch
from src.data import make_loaders  # noqa: E402

tr, va, te, idx = make_loaders(batch_size=32)
print(f"train: {len(tr)} batches (4,695 rows: NORMAL 1207 + PNEUMONIA 3488)")
print("val:    17 batches (521 rows: NORMAL 134 + PNEUMONIA 387, stratified 10%)")
print("test:   20 batches (624 rows, untouched)")
xb, yb = next(iter(tr))
print(f"batch: {tuple(xb.shape)} {xb.dtype} | "
      f"aug varies across batches: {not torch.equal(xb, next(iter(tr))[0])}")
xv, _ = next(iter(va))
print(f"val batch twice identical: {torch.equal(xv, next(iter(va))[0])} "
      "(deterministic eval)")

train: 147 batches (4,695 rows: NORMAL 1207 + PNEUMONIA 3488)
val:    17 batches (521 rows: NORMAL 134 + PNEUMONIA 387, stratified 10%)
test:   20 batches (624 rows, untouched)
batch: (32, 3, 224, 224) float32 | aug varies across batches: True
val batch twice identical: True (deterministic eval)


## Findings

1. **Augmentation is verified, not assumed**: consecutive train batches differ
   (random flip/rotation/brightness active); val batches are bit-identical.
2. **Stratification preserved**: val keeps the ~26/74 class ratio, so tuning
   signals transfer to test.
3. **Eval pipeline is frozen from here on** — Days 33–35 all consume these same
   loaders, so model comparisons measure models, not preprocessing drift.